# Winning Combinations

**Why**: Test if top ideas have synergy when combined, or if they interfere.

**Combos**:
1. **revise + JEPA** — visual ceiling push (cifar10, mazes)
2. **revise + sparsity** — sort ceiling via two mechanisms (sort, mazes)
3. **JEPA + sparsity** — representation regularization synergy (cifar10, sort)
4. **revise + JEPA + sparsity** — full stack (cifar10, sort)

**Hardware**: 1 machine x 8 GPUs. 24 runs, ~8h.

Run from the `paper/` directory.

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import make_combo, run_all, status, collect, plot_delta_bars
%matplotlib inline

In [ ]:
COMBOS = [
    ('revise+jepa',     make_combo(['cifar10', 'mazes'], [0, 1, 2], use_revise=True, use_jepa=True)),
    ('revise+sparsity', make_combo(['sort', 'mazes'],    [0, 1, 2], use_revise=True, use_sparsity=True)),
    ('jepa+sparsity',   make_combo(['cifar10', 'sort'],  [0, 1, 2], use_jepa=True, use_sparsity=True)),
    ('full_stack',      make_combo(['cifar10', 'sort'],  [0, 1, 2], use_revise=True, use_jepa=True, use_sparsity=True)),
]

exps = []
for name, group in COMBOS:
    print(f'{name:20s}: {len(group)} runs')
    exps.extend(group)
print(f'\nTotal: {len(exps)} experiments')

## Step 1 — Dry run

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/04_combos', dry_run=True)

## Step 2 — Run training

Uncomment to launch (~8h).

In [ ]:
# done, failed = run_all(exps, gpus=8, log_root='logs/deep/04_combos')

In [ ]:
status('logs/deep/04_combos')

## Step 3 — Results

Compare combos vs single-idea baselines vs paper baseline.

In [ ]:
df = collect('logs/deep/04_combos')
if df.empty:
    print('No results yet.')
else:
    df['combo'] = df['name'].apply(
        lambda n: '+'.join([p for p in ['revise', 'jepa', 'spar'] if p in n]) or 'baseline')
    print(df[['name', 'task', 'combo', 'best_acc', 'delta']].to_string(index=False))
    plot_delta_bars(df, 'Winning combos vs baseline', 'figures/04_combos_delta.png')

In [ ]:
# Grouped bar chart: combo x task
if not df.empty and 'best_acc' in df and df['best_acc'].notna().any():
    import matplotlib.pyplot as plt
    import numpy as np
    from exp_runner import BASELINE_ACC

    agg = df.dropna(subset=['best_acc']).groupby(['combo', 'task'])['best_acc'].agg(['mean', 'std']).reset_index()
    tasks_present = sorted(agg['task'].unique())
    combos_present = sorted(agg['combo'].unique())
    x = np.arange(len(combos_present))
    width = 0.8 / max(len(tasks_present), 1)

    fig, ax = plt.subplots(figsize=(12, 5.5))
    for i, task in enumerate(tasks_present):
        vals = []
        for cmb in combos_present:
            row = agg[(agg['combo'] == cmb) & (agg['task'] == task)]
            vals.append(row['mean'].values[0] * 100 if not row.empty else 0)
        ax.bar(x + i * width - 0.4 + width / 2, vals, width, label=task, edgecolor='black', lw=0.4)
        for j, v in enumerate(vals):
            bl = BASELINE_ACC.get(task, 0) * 100
            if v > 0:
                ax.text(x[j] + i * width - 0.4 + width / 2, v + 0.8, f'{v:.1f}',
                        ha='center', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(combos_present, rotation=15, fontsize=10)
    ax.set_ylabel('best test acc (%)')
    ax.set_title('Winning combos: grouped by task (dashed = paper baseline)')
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.2)

    for i, task in enumerate(tasks_present):
        bl = BASELINE_ACC.get(task)
        if bl:
            ax.axhline(bl * 100, color=['#1f77b4','#ff7f0e','#9467bd','#2ca02c'][i % 4],
                       ls=':', alpha=0.4, lw=1)

    fig.tight_layout()
    fig.savefig('figures/04_combos_grouped.png', dpi=150, bbox_inches='tight')
    plt.show()